# Context Cache

대량 텍스트/파일을 캐싱하여 반복 호출 시 비용 절감.

| 모델 | 최소 토큰 |
|------|----------|
| Flash 계열 | 1,024 |
| Pro 계열 | 4,096 |

`create_cache()` → `get_cache()` → `update_cache()` → `delete_cache()` 전체 라이프사이클.

In [1]:
from llm_manager._internal import handler_client as llm_handler
from llm_manager._internal.handler_client import (
    call,
    upload_file,
    create_cache,
    get_cache,
    update_cache,
    list_caches,
    delete_cache,
    delete_file,
)

# llm_handler.configure(base_url="http://localhost:8100")

---

## 1. 텍스트 캐싱

대량 텍스트를 캐싱하고, 캐시를 참조하여 여러 번 질문.

In [ ]:
# 캐시 생성 (최소 토큰: Flash 1,024 / Pro 4,096)
large_text = "ESG 보고서 내용..." * 5000  # 충분한 길이

cache = create_cache(
    large_text,  # str | list[str | FileRef | Part] — call()과 동일
    model="gemini-3-flash-preview",
    system="ESG 전문가로서 답변하세요.",
    display_name="text-cache-test",
    ttl=300,  # 5분
)
print(f"name: {cache.name}")
print(f"total_token_count: {cache.total_token_count}")
print(f"expire_time: {cache.expire_time}")

In [ ]:
# 캐시 참조하여 질문 (토큰 비용 절감)
r1 = call("환경 항목 평가해줘", cached_content=cache.name)
print(f"[질의 1] {r1.text[:100]}...")

r2 = call("거버넌스 항목 평가해줘", cached_content=cache.name)
print(f"[질의 2] {r2.text[:100]}...")

In [ ]:
# 캐시 상세 조회
info = get_cache(cache.name)
print(f"model: {info.model}")
print(f"display_name: {info.display_name}")
print(f"update_time: {info.update_time}")

# TTL 연장 (300초 → 600초)
updated = update_cache(cache.name, ttl=600)
print(f"\n[TTL 연장] expire_time: {updated.expire_time}")

# 캐시 목록
caches = list_caches()
print(f"\n전체 캐시 수: {len(caches)}개")
for c in caches:
    print(f"  {c.display_name}: {c.name} (tokens={c.total_token_count})")

In [ ]:
# 정리
delete_cache(cache.name)
print("텍스트 캐시 삭제 완료")

---

## 2. 파일 캐싱 (오디오)

Files API로 업로드한 파일을 캐싱. 동일 파일로 반복 질의 시 비용 절감.

In [9]:
# 1. 파일 업로드
AUDIO_PATH = "test_audio.m4a"  # quickstarts/ 폴더 내 테스트 파일

file_ref = upload_file(AUDIO_PATH, display_name="테스트 오디오")
print(f"file: {file_ref.name}, display name:{file_ref.display_name} ({file_ref.size_bytes} bytes)")
print(f"mime_type: {file_ref.mime_type}")

file: files/90ex3etceouy, display name:upload (4006244 bytes)
mime_type: audio/mp4


In [10]:
# 2. 파일을 contents에 직접 전달 — call()과 동일한 방식
from llm_manager._internal.handler_client import Part

cache = create_cache(
    [Part.from_uri(file_ref.uri, file_ref.mime_type)],  # FileRef URI 참조
    model="gemini-3-flash-preview",
    display_name="audio-cache-test",
    ttl=300,
)
print(f"cache name(id 역할): {cache.name}")
print(f"cache display: {cache.display_name}")
print(f"total_token_count: {cache.total_token_count}")

cache name(id 역할): cachedContents/wl9850kbmxqhbkv9g1crlple0mx5k4k3jq18q1w6
cache display: audio-cache-test
total_token_count: 7913


In [4]:
# 3. 캐시 참조하여 반복 질의
r1 = call("이 오디오의 내용을 요약해줘", cached_content=cache.name)
print(f"[요약] {r1.text[:200]}...")

r2 = call("화자가 몇 명이야?", cached_content=cache.name)
print(f"[화자] {r2.text[:200]}...")

[요약] 해당 오디오의 내용을 요약하면 다음과 같습니다.

1. **코람코와의 협상 상황:** 코람코 측에서 원래 오늘 승인(OK)을 해주기로 했으나, 현장 방문 등의 이유로 내일로 결정을 미룬 상태입니다. 코람코가 후단 자금 부분을 해결해 줄 수 있는지 여부가 핵심입니다.
2. **자금 준비 상태:** 현재 계약금으로 약 60억 원 정도가 준비되어 있으나, 나머지...
[화자] 네, 해당 통화에는 남성 2명이 대화를 나누고 있습니다. 한 명은 '사장님'이라고 불리는 분이며, 다른 한 명은 그에게 상황을 묻고 보고하는 '대표님'이라고 불리는 분입니다. 두 화자는 코란코와의 계약, 계약금 준비, 키움증권과의 연락 등 비즈니스와 관련된 내용을 논의하고 있습니다. 

따라서 총 **2명**의 화자가 있습니다....


In [5]:
# 4. 캐시 조회 + TTL 연장
info = get_cache(cache.name)
print(f"expire_time (원래): {info.expire_time}")

updated = update_cache(cache.name, ttl=600)
print(f"expire_time (연장): {updated.expire_time}")

expire_time (원래): 2026-04-16 05:19:27.483285+00:00
expire_time (연장): 2026-04-16 05:25:10.837598+00:00


In [6]:
# 5. 정리 (캐시 + 파일)
delete_cache(cache.name)
print("캐시 삭제 완료")

delete_file(file_ref.name)
print("파일 삭제 완료")

캐시 삭제 완료
파일 삭제 완료


In [ ]:
info = get_cache(cache.name)
print(f"expire_time (원래): {info.expire_time}") # 에러 뜨는게 정상 (403)

HandlerError: [provider] Gemini 캐시 조회 에러: 403 PERMISSION_DENIED. {'error': {'code': 403, 'message': 'CachedContent not found (or permission denied)', 'status': 'PERMISSION_DENIED'}} (403)